# Bab 7: Metode Kernel dan Proses Gaussian

Notebook pendamping buku *Pengenalan Pola dan Machine Learning: Pendekatan Probabilistik dengan Python*.

Jalankan sel dari atas ke bawah (**Runtime → Run all** di Colab). Sel pertama menyiapkan folder kode dan modul data sintetis; setiap sel berikutnya adalah satu listing dari buku, dengan nomor penanda `# (1)`, `# (2)`, … yang dijelaskan di bawah listing pada buku. Angka keluaran bisa sedikit berbeda dari buku bila versi pustakanya berbeda (lihat Lampiran A).

In [ ]:
# Menyiapkan folder kode/ dan modul pendukung bab ini
import os, pathlib
BERKAS = {
"bab07/data_pm25.py": "\"\"\"Medan PM2,5 sintetis di wilayah kota berukuran 20 x 20 km (ilustrasi).\n\nKonsentrasi latar ditambah beberapa 'titik panas' (kawasan industri, jalan\ntol, simpul kemacetan) dan derau sensor murah. Satuan mikrogram per m^3.\n\"\"\"\nimport numpy as np\n\nPANAS = [(5, 14, 25, 2.5), (13, 6, 30, 3.0), (15, 15, 18, 4.0), (8, 4, 12, 2.0)]\n\ndef medan(x, y):\n    z = 35 + 0.6 * x                                   # latar, naik ke timur\n    for cx, cy, a, s in PANAS:\n        z = z + a * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * s**2))\n    return z\n\ndef sensor(n=40, sd=3.0, seed=0):\n    rng = np.random.default_rng(seed)\n    X = rng.uniform(0, 20, size=(n, 2))\n    return X, medan(X[:, 0], X[:, 1]) + rng.normal(0, sd, n)\n\ndef jam(n_hari=3, seed=1):\n    \"\"\"Satu sensor: PM2,5 per jam dengan pola harian dan jam-jam hilang.\"\"\"\n    rng = np.random.default_rng(seed)\n    t = np.arange(24 * n_hari, dtype=float)\n    f = 45 + 15 * np.cos(2 * np.pi * (t - 7) / 24) + 8 * np.sin(2 * np.pi * t / 60)\n    ada = ~((t >= 30) & (t < 44))                      # sensor mati 14 jam\n    ada &= rng.random(t.size) < 0.6                    # dan data bolong acak\n    return t[ada], f[ada] + rng.normal(0, 4, ada.sum()), t, f\n",
"bab07/dual_ridge.py": "import numpy as np\nimport time\n\nrng = np.random.default_rng(0)\nN, M = 2000, 30                                    # banyak data, banyak fitur\nPhi = rng.normal(size=(N, M)); t = Phi @ rng.normal(size=M) + rng.normal(0, 0.5, N)\nlam = 1.0\n\nt0 = time.perf_counter()                           # bentuk primal: M x M  #<1>\nw = np.linalg.solve(Phi.T @ Phi + lam * np.eye(M), Phi.T @ t)\nt_primal = time.perf_counter() - t0\n\nt0 = time.perf_counter()                           # bentuk dual: N x N  #<2>\nK = Phi @ Phi.T                                    # matriks Gram k(x_n, x_m)\na = np.linalg.solve(K + lam * np.eye(N), t)\nt_dual = time.perf_counter() - t0\n\nphi_baru = rng.normal(size=(5, M))\nprint(\"prediksi primal:\", np.round(phi_baru @ w, 4))\nprint(\"prediksi dual  :\", np.round(phi_baru @ Phi.T @ a, 4))   # sum_n a_n k(x, x_n)  #<3>\nprint(f\"waktu primal {t_primal*1e3:.1f} ms (matriks {M}x{M}), \"\n      f\"dual {t_dual*1e3:.1f} ms (matriks {N}x{N})\")\n",
"bab07/gp_hiperparameter.py": "import numpy as np\nfrom scipy.optimize import minimize\nfrom data_pm25 import jam\n\nx, t, _, _ = jam()\nt = t - t.mean()\n\ndef log_marginal(log_theta):                      # ln p(t | theta)  #<1>\n    amp, ell, sd = np.exp(log_theta)\n    C = amp**2 * np.exp(-(x[:, None] - x[None, :])**2 / (2 * ell**2)) + sd**2 * np.eye(len(x))\n    L = np.linalg.cholesky(C)\n    alpha = np.linalg.solve(L.T, np.linalg.solve(L, t))\n    kecocokan = -0.5 * t @ alpha                   # seberapa cocok dengan data\n    kompleksitas = -np.sum(np.log(np.diag(L)))     # -1/2 ln|C|  #<2>\n    return kecocokan + kompleksitas - 0.5 * len(x) * np.log(2 * np.pi), kecocokan, kompleksitas\n\nprint(\" skala (jam)  ln p(t)   kecocokan  kompleksitas\")\nfor ell in (0.5, 2, 4, 8, 30):\n    lp, a, b = log_marginal(np.log([15, ell, 4]))\n    print(f\"{ell:11.1f} {lp:9.1f} {a:10.1f} {b:13.1f}\")\n\nhasil = minimize(lambda z: -log_marginal(z)[0], np.log([10, 3, 2]))   #<3>\namp, ell, sd = np.exp(hasil.x)\nprint(f\"optimum: amplitudo {amp:.1f}, skala {ell:.1f} jam, sd derau {sd:.1f} \"\n      \"(sebenarnya 4.0)\")\nprint(f\"ln p(t) di optimum = {-hasil.fun:.1f}\")\n",
"bab07/gp_regresi.py": "import numpy as np\nfrom data_pm25 import jam\n\nx, t, x_semua, f_semua = jam()                     # jam pengamatan, PM2,5\n\ndef kernel(a, b, theta=(20.0, 4.0)):               # RBF: amplitudo^2, skala  #<1>\n    amp2, ell = theta\n    return amp2**2 * np.exp(-(a[:, None] - b[None, :])**2 / (2 * ell**2))\n\ndef gp_prediksi(x, t, x_baru, theta, sd_derau):\n    m = t.mean()                                   # rerata konstanta\n    C = kernel(x, x, theta) + sd_derau**2 * np.eye(len(x))     #<2>\n    L = np.linalg.cholesky(C)                      # stabil dan cepat\n    alpha = np.linalg.solve(L.T, np.linalg.solve(L, t - m))\n    k = kernel(x, x_baru, theta)\n    rerata = m + k.T @ alpha                       # m(x*) = k^T C^-1 t  #<3>\n    v = np.linalg.solve(L, k)\n    varians = theta[0]**2 - np.sum(v**2, axis=0)     # c - k^T C^-1 k\n    return rerata, np.sqrt(varians)                  # sd fungsi f (tanpa derau)\n\nprint(f\"pengamatan: {len(x)} dari {len(x_semua)} jam\")\nr, s = gp_prediksi(x, t, x_semua, (20.0, 4.0), 4.0)\nfor jam_ in (20, 36, 40, 60):                      # 36 dan 40: sensor mati\n    print(f\"jam {jam_:2d}: prediksi {r[jam_]:5.1f} +- {2*s[jam_]:4.1f} | sebenarnya {f_semua[jam_]:5.1f}\")\ncakup = np.mean(np.abs(f_semua - r) < 2 * s)\nprint(f\"fungsi sebenarnya di dalam pita +-2sd: {cakup:.0%} dari semua jam\")\n",
"bab07/gp_spasial.py": "import numpy as np\nfrom sklearn.gaussian_process import GaussianProcessRegressor\nfrom sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel\nfrom data_pm25 import sensor, medan\n\nX, t = sensor(n=40)                                # 40 sensor murah\ng = np.linspace(0, 20, 81)\nG = np.array([(a, b) for b in g for a in g])       # grid peta 81 x 81\nf_benar = medan(G[:, 0], G[:, 1])\n\nkern = ConstantKernel(100) * RBF(length_scale=3.0) + WhiteKernel(5.0)   #<1>\ngp = GaussianProcessRegressor(kern, normalize_y=True, n_restarts_optimizer=5,\n                              random_state=0).fit(X, t)                   #<2>\nprint(\"kernel hasil optimasi:\\n \", gp.kernel_)\nm, s = gp.predict(G, return_std=True)              # rerata dan sd di setiap sel  #<3>\n\ndef idw(G, X, t, p=2):                             # pembanding: inverse distance weighting\n    d = np.linalg.norm(G[:, None] - X[None], axis=2) + 1e-9\n    w = d**-p\n    return (w @ t) / w.sum(1)\n\nfor nama, pred in [(\"rata-rata sensor\", np.full(len(G), t.mean())),\n                   (\"IDW\", idw(G, X, t)), (\"proses Gaussian\", m)]:\n    print(f\"{nama:17s}: RMSE peta {np.sqrt(np.mean((pred - f_benar)**2)):5.2f} ug/m3\")\ns_f = np.sqrt(np.maximum(s**2 - gp.kernel_.k2.noise_level * np.var(t), 0))  # tanpa derau\nprint(f\"medan sebenarnya di dalam +-2sd: {np.mean(np.abs(f_benar - m) < 2 * s_f):.0%} sel\")\n"
}
akar = pathlib.Path('pola-ml/kode').resolve()
for nama, isi in BERKAS.items():
    p = akar / nama; p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(isi, encoding='utf-8')
os.chdir(akar / 'bab07')
print('folder kerja:', os.getcwd())

## Listing 7.1: Ridge dalam bentuk primal dan dual memberi prediksi yang sama

Berkas: `kode/bab07/dual_ridge.py`

In [ ]:
import numpy as np
import time

rng = np.random.default_rng(0)
N, M = 2000, 30                                    # banyak data, banyak fitur
Phi = rng.normal(size=(N, M)); t = Phi @ rng.normal(size=M) + rng.normal(0, 0.5, N)
lam = 1.0

t0 = time.perf_counter()                           # bentuk primal: M x M  # (1)
w = np.linalg.solve(Phi.T @ Phi + lam * np.eye(M), Phi.T @ t)
t_primal = time.perf_counter() - t0

t0 = time.perf_counter()                           # bentuk dual: N x N  # (2)
K = Phi @ Phi.T                                    # matriks Gram k(x_n, x_m)
a = np.linalg.solve(K + lam * np.eye(N), t)
t_dual = time.perf_counter() - t0

phi_baru = rng.normal(size=(5, M))
print("prediksi primal:", np.round(phi_baru @ w, 4))
print("prediksi dual  :", np.round(phi_baru @ Phi.T @ a, 4))   # sum_n a_n k(x, x_n)  # (3)
print(f"waktu primal {t_primal*1e3:.1f} ms (matriks {M}x{M}), "
      f"dual {t_dual*1e3:.1f} ms (matriks {N}x{N})")

## Listing 7.2: Regresi proses Gaussian dari nol untuk PM$_2,5$ per jam

Berkas: `kode/bab07/gp_regresi.py`

In [ ]:
import numpy as np
from data_pm25 import jam

x, t, x_semua, f_semua = jam()                     # jam pengamatan, PM2,5

def kernel(a, b, theta=(20.0, 4.0)):               # RBF: amplitudo^2, skala  # (1)
    amp2, ell = theta
    return amp2**2 * np.exp(-(a[:, None] - b[None, :])**2 / (2 * ell**2))

def gp_prediksi(x, t, x_baru, theta, sd_derau):
    m = t.mean()                                   # rerata konstanta
    C = kernel(x, x, theta) + sd_derau**2 * np.eye(len(x))     # (2)
    L = np.linalg.cholesky(C)                      # stabil dan cepat
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, t - m))
    k = kernel(x, x_baru, theta)
    rerata = m + k.T @ alpha                       # m(x*) = k^T C^-1 t  # (3)
    v = np.linalg.solve(L, k)
    varians = theta[0]**2 - np.sum(v**2, axis=0)     # c - k^T C^-1 k
    return rerata, np.sqrt(varians)                  # sd fungsi f (tanpa derau)

print(f"pengamatan: {len(x)} dari {len(x_semua)} jam")
r, s = gp_prediksi(x, t, x_semua, (20.0, 4.0), 4.0)
for jam_ in (20, 36, 40, 60):                      # 36 dan 40: sensor mati
    print(f"jam {jam_:2d}: prediksi {r[jam_]:5.1f} +- {2*s[jam_]:4.1f} | sebenarnya {f_semua[jam_]:5.1f}")
cakup = np.mean(np.abs(f_semua - r) < 2 * s)
print(f"fungsi sebenarnya di dalam pita +-2sd: {cakup:.0%} dari semua jam")

## Listing 7.3: Marginal likelihood sebagai fungsi skala panjang

Berkas: `kode/bab07/gp_hiperparameter.py`

In [ ]:
import numpy as np
from scipy.optimize import minimize
from data_pm25 import jam

x, t, _, _ = jam()
t = t - t.mean()

def log_marginal(log_theta):                      # ln p(t | theta)  # (1)
    amp, ell, sd = np.exp(log_theta)
    C = amp**2 * np.exp(-(x[:, None] - x[None, :])**2 / (2 * ell**2)) + sd**2 * np.eye(len(x))
    L = np.linalg.cholesky(C)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, t))
    kecocokan = -0.5 * t @ alpha                   # seberapa cocok dengan data
    kompleksitas = -np.sum(np.log(np.diag(L)))     # -1/2 ln|C|  # (2)
    return kecocokan + kompleksitas - 0.5 * len(x) * np.log(2 * np.pi), kecocokan, kompleksitas

print(" skala (jam)  ln p(t)   kecocokan  kompleksitas")
for ell in (0.5, 2, 4, 8, 30):
    lp, a, b = log_marginal(np.log([15, ell, 4]))
    print(f"{ell:11.1f} {lp:9.1f} {a:10.1f} {b:13.1f}")

hasil = minimize(lambda z: -log_marginal(z)[0], np.log([10, 3, 2]))   # (3)
amp, ell, sd = np.exp(hasil.x)
print(f"optimum: amplitudo {amp:.1f}, skala {ell:.1f} jam, sd derau {sd:.1f} "
      "(sebenarnya 4.0)")
print(f"ln p(t) di optimum = {-hasil.fun:.1f}")

## Listing 7.4: Peta PM$_2,5$ dari 40 sensor dengan proses Gaussian

Berkas: `kode/bab07/gp_spasial.py`

In [ ]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from data_pm25 import sensor, medan

X, t = sensor(n=40)                                # 40 sensor murah
g = np.linspace(0, 20, 81)
G = np.array([(a, b) for b in g for a in g])       # grid peta 81 x 81
f_benar = medan(G[:, 0], G[:, 1])

kern = ConstantKernel(100) * RBF(length_scale=3.0) + WhiteKernel(5.0)   # (1)
gp = GaussianProcessRegressor(kern, normalize_y=True, n_restarts_optimizer=5,
                              random_state=0).fit(X, t)                   # (2)
print("kernel hasil optimasi:\n ", gp.kernel_)
m, s = gp.predict(G, return_std=True)              # rerata dan sd di setiap sel  # (3)

def idw(G, X, t, p=2):                             # pembanding: inverse distance weighting
    d = np.linalg.norm(G[:, None] - X[None], axis=2) + 1e-9
    w = d**-p
    return (w @ t) / w.sum(1)

for nama, pred in [("rata-rata sensor", np.full(len(G), t.mean())),
                   ("IDW", idw(G, X, t)), ("proses Gaussian", m)]:
    print(f"{nama:17s}: RMSE peta {np.sqrt(np.mean((pred - f_benar)**2)):5.2f} ug/m3")
s_f = np.sqrt(np.maximum(s**2 - gp.kernel_.k2.noise_level * np.var(t), 0))  # tanpa derau
print(f"medan sebenarnya di dalam +-2sd: {np.mean(np.abs(f_benar - m) < 2 * s_f):.0%} sel")